In [1]:
import pandas as pd


# 1. Silnik wnioskowania w przód (Wariant 13: Przypadki skrajne)
def wnioskowanie_w_przod(pacjent):
    sbp = int(pacjent["systolic_bp"])
    dbp = int(pacjent["diastolic_bp"])
    glukoza = int(pacjent["glucose"])

    wnioski = []
    logika = []

    # Reguła 1: Przypadek skrajny (Maksymalna siła reguł)
    if sbp >= 180 or dbp >= 120:
        wnioski.append("Przełom nadciśnieniowy (Kryza)")
        logika.append(
            f"PRZYPADEK SKRAJNY: Maksymalna siła reguły dla krytycznego ciśnienia (SBP={sbp} / DBP={dbp})"
        )

    # Reguła 2: Reguły rozmyte (Pełna aktywacja zbioru "Wysokie")
    elif sbp >= 140 or dbp >= 90:
        wnioski.append("Nadciśnienie tętnicze")
        logika.append(
            f"PEŁNA AKTYWACJA: Ciśnienie zakwalifikowane jako WYSOKIE (SBP={sbp} / DBP={dbp})"
        )

    # Reguła dodatkowa: Glukoza (Hiperglikemia)
    if glukoza > 125:
        wnioski.append("Hiperglikemia")
        logika.append(
            f"AKTYWACJA REGUŁY: Glukoza przekracza normę (Glukoza={glukoza} mg/dl)"
        )

    if not wnioski:
        wnioski.append("Norma")
        logika.append("Brak aktywacji reguł chorobowych – parametry w normie")

    return str(", ".join(wnioski)), str(" | ".join(logika))


# 2. Wczytanie, oczyszczenie danych z pliku CSV i wnioskowanie
try:
    df = pd.read_csv("dane_pacjentow.csv")

    # Czyszczenie i rzutowanie typów
    for col in ["glucose", "systolic_bp", "diastolic_bp"]:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.replace(r"[^\d]", "", regex=True)
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # 3. Uruchomienie silnika
    wyniki = []
    for _, pacjent in df.iterrows():
        diagnoza, wyjasnienie = wnioskowanie_w_przod(pacjent)
        wyniki.append(
            {
                "ID": pacjent["patient_id"],
                "Diagnoza": diagnoza,
                "Wyjaśnienie systemowe": wyjasnienie,
            }
        )

    df_wyniki = pd.DataFrame(wyniki)
    pd.set_option("display.max_colwidth", None)
    print(df_wyniki.to_string(index=False))

except FileNotFoundError:
    print("Błąd: Nie znaleziono pliku 'dane_pacjentow.csv' w katalogu skryptu.")


 ID                                      Diagnoza                                                                                                                                   Wyjaśnienie systemowe
P01          Nadciśnienie tętnicze, Hiperglikemia             PEŁNA AKTYWACJA: Ciśnienie zakwalifikowane jako WYSOKIE (SBP=176 / DBP=83) | AKTYWACJA REGUŁY: Glukoza przekracza normę (Glukoza=168 mg/dl)
P02                         Nadciśnienie tętnicze                                                                              PEŁNA AKTYWACJA: Ciśnienie zakwalifikowane jako WYSOKIE (SBP=148 / DBP=98)
P03                                         Norma                                                                                                   Brak aktywacji reguł chorobowych – parametry w normie
P04          Nadciśnienie tętnicze, Hiperglikemia             PEŁNA AKTYWACJA: Ciśnienie zakwalifikowane jako WYSOKIE (SBP=167 / DBP=91) | AKTYWACJA REGUŁY: Glukoza przekracza normę (Glukoza=1